# Using the DuckDB Network Database

This notebook shows how to query the enriched road network database produced by the
pipeline (notebooks 1–5) and provides reusable helper functions for common tasks.

## What is in the DuckDB?

| Table | Schema | Content |
|---|---|---|
| `edges` | `driving` / `walking` / `cycling` | Road segments with geometry, speed, cost, H3 index, congestion |
| `nodes` | `driving` / `walking` / `cycling` | Network nodes (intersections) |
| `edge_graph` | `driving` / `walking` / `cycling` | Adjacency list for routing |
| `turn_restrictions` | `driving` | Turn constraints from OSM |
| `runs` | `main` | Metadata per Mapbox fetch (timestamp, tile count) |
| `traffic_segments` | `main` | Raw Mapbox traffic segments per run |
| `edge_congestion_history` | `main` | Time-series: congestion per edge per run |
| `boundary_cells` | `main` | H3 hexagons covering the boundary |

## Key columns in `driving.edges`

| Column | Type | Description |
|---|---|---|
| `edge_id` | INTEGER | Unique edge ID |
| `highway` | VARCHAR | OSM road type (`residential`, `secondary`, …) |
| `name` | VARCHAR | Street name |
| `oneway` | VARCHAR | Directionality |
| `length_m` | DOUBLE | Length in metres |
| `maxspeed_kmh` | DOUBLE | Speed limit |
| `cost_s` | DOUBLE | Travel time in seconds |
| `from_cell` | VARCHAR | H3 cell of start node |
| `to_cell` | VARCHAR | H3 cell of end node |
| `geometry` | GEOMETRY | LineString (WGS-84) |
| `congestion` | VARCHAR | Latest traffic: `low` / `moderate` / `heavy` / `severe` / `no data` |

In [ ]:
%%time
import duckdb
import geopandas as gpd
import pandas as pd
import folium
import yaml
from pathlib import Path

# ── Configuration ─────────────────────────────────────────────────────────
NAME = 'sodermalm'
# ─────────────────────────────────────────────────────────────────────────

CONFIG_PATH = Path(f'../config/{NAME}.yaml')
if CONFIG_PATH.exists():
    with open(CONFIG_PATH) as f:
        cfg = yaml.safe_load(f)
    DB_PATH = Path(cfg['output_path']) / f"{cfg['name']}.duckdb"
else:
    DB_PATH = Path(f'../db/{NAME}.duckdb')

print(f'Database: {DB_PATH}')
print(f'Exists  : {DB_PATH.exists()}  ({DB_PATH.stat().st_size/1_048_576:.1f} MB)' if DB_PATH.exists() else 'NOT FOUND')

---
## Helper functions

These functions wrap common DuckDB queries into a clean Python API.
All functions accept an open DuckDB connection and return a pandas DataFrame or GeoDataFrame.

In [ ]:
%%time
from shapely import wkt

CONGESTION_COLORS = {
    'low':      '#00c800',
    'moderate': '#ffa500',
    'heavy':    '#ff4500',
    'severe':   '#8b0000',
    'no data':  '#cccccc',
}


def connect(db_path=DB_PATH):
    """Open a read-only connection to the DuckDB."""
    con = duckdb.connect(str(db_path), read_only=True)
    con.execute('LOAD spatial')
    return con


def list_tables(con):
    """Return a DataFrame of all user tables grouped by schema."""
    return con.execute("""
        SELECT table_schema  AS schema,
               table_name    AS table_name,
               (SELECT count(*) FROM information_schema.columns c
                WHERE c.table_schema = t.table_schema
                  AND c.table_name  = t.table_name) AS columns
        FROM information_schema.tables t
        WHERE table_schema NOT IN ('pg_catalog','information_schema')
        ORDER BY schema, table_name
    """).df()


def to_gdf(df, wkt_col='wkt_geom', crs='EPSG:4326'):
    """Convert a DataFrame with a WKT geometry column to a GeoDataFrame."""
    return gpd.GeoDataFrame(
        df.drop(columns=[wkt_col]),
        geometry=gpd.GeoSeries.from_wkt(df[wkt_col]),
        crs=crs,
    )


# ── Edge queries ──────────────────────────────────────────────────────────

def get_edges(con, mode='driving', highway=None, congestion=None,
              name=None, limit=None):
    """
    Fetch road edges as a GeoDataFrame.

    Parameters
    ----------
    mode       : 'driving' | 'walking' | 'cycling'
    highway    : filter by OSM road type, e.g. 'residential'
    congestion : filter by congestion level, e.g. 'heavy'
    name       : partial street name match (case-insensitive)
    limit      : max rows to return
    """
    filters = []
    if highway:    filters.append(f"highway = '{highway}'")
    if congestion: filters.append(f"congestion = '{congestion}'")
    if name:       filters.append(f"lower(name) LIKE '%{name.lower()}%'")
    where = ('WHERE ' + ' AND '.join(filters)) if filters else ''
    lim   = f'LIMIT {limit}' if limit else ''

    df = con.execute(f"""
        SELECT edge_id, highway, name, oneway, length_m,
               maxspeed_kmh, cost_s, from_cell, to_cell, congestion,
               ST_AsText(geometry) AS wkt_geom
        FROM {mode}.edges
        {where}
        ORDER BY edge_id
        {lim}
    """).df()
    return to_gdf(df)


def get_edge_by_id(con, edge_id, mode='driving'):
    """Fetch a single edge by its ID."""
    return get_edges(con, mode=mode).query(f'edge_id == {edge_id}')


def get_network_stats(con, mode='driving'):
    """Summary statistics for the road network."""
    return con.execute(f"""
        SELECT highway,
               count(*)                      AS edges,
               round(sum(length_m)/1000, 1)  AS km,
               round(avg(maxspeed_kmh), 0)   AS avg_speed_kmh,
               round(avg(cost_s), 1)         AS avg_cost_s
        FROM {mode}.edges
        GROUP BY highway
        ORDER BY edges DESC
    """).df()


# ── Congestion queries ────────────────────────────────────────────────────

def get_congestion_summary(con, mode='driving'):
    """Current congestion distribution — edge count and km per level."""
    return con.execute(f"""
        SELECT congestion,
               count(*)                     AS edges,
               round(sum(length_m)/1000, 1) AS km,
               round(count(*) * 100.0 / sum(count(*)) OVER (), 1) AS pct
        FROM {mode}.edges
        GROUP BY congestion
        ORDER BY km DESC
    """).df()


def get_congestion_history(con, edge_id=None, road_name=None):
    """
    Time-series congestion for a specific edge or road name.

    Supply either edge_id (exact) or road_name (partial match).
    Returns one row per (edge, run) combination.
    """
    if edge_id:
        edge_filter = f'h.edge_id = {edge_id}'
    elif road_name:
        edge_filter = f"lower(e.name) LIKE '%{road_name.lower()}%'"
    else:
        raise ValueError('Supply either edge_id or road_name')

    return con.execute(f"""
        SELECT r.run_id, r.fetched_at, r.boundary_name,
               h.edge_id, e.name, e.highway,
               h.congestion, h.matched_at
        FROM edge_congestion_history h
        JOIN runs r ON h.run_id = r.run_id
        JOIN driving.edges e ON h.edge_id = e.edge_id
        WHERE {edge_filter}
        ORDER BY r.fetched_at, h.edge_id
    """).df()


# ── Run / history queries ─────────────────────────────────────────────────

def get_runs(con):
    """List all pipeline fetch runs with their metadata."""
    return con.execute("""
        SELECT r.run_id, r.boundary_name, r.zoom,
               r.fetched_at, r.n_tiles, r.n_segments,
               count(h.edge_id) AS edges_matched
        FROM runs r
        LEFT JOIN edge_congestion_history h ON h.run_id = r.run_id
        GROUP BY 1,2,3,4,5,6
        ORDER BY r.fetched_at
    """).df()


# ── H3 / spatial queries ──────────────────────────────────────────────────

def get_boundary_cells(con):
    """Fetch H3 boundary cells as a GeoDataFrame."""
    df = con.execute("""
        SELECT h3_id, resolution, ST_AsText(geometry) AS wkt_geom
        FROM boundary_cells
    """).df()
    return to_gdf(df)


def get_edges_in_cell(con, h3_id, mode='driving'):
    """Fetch all edges whose source H3 cell matches the given h3_id."""
    df = con.execute(f"""
        SELECT edge_id, highway, name, congestion, length_m,
               ST_AsText(geometry) AS wkt_geom
        FROM {mode}.edges
        WHERE from_cell = '{h3_id}'
    """).df()
    return to_gdf(df)


def get_congestion_by_cell(con, mode='driving'):
    """
    Dominant congestion level per H3 cell.
    Useful for kepler.gl / heat-map style visualisation.
    """
    return con.execute(f"""
        SELECT from_cell AS h3_id,
               count(*)                     AS edges,
               round(sum(length_m)/1000, 2) AS km,
               mode() WITHIN GROUP (ORDER BY congestion) AS dominant_congestion
        FROM {mode}.edges
        WHERE from_cell IS NOT NULL
        GROUP BY from_cell
        ORDER BY edges DESC
    """).df()


# ── Visualisation ─────────────────────────────────────────────────────────

def plot_edges(gdf, color_col='congestion', zoom=14, tiles='OpenStreetMap'):
    """
    Display a GeoDataFrame of edges on a folium map.
    Edges are colored by their congestion level if color_col='congestion'.
    """
    lines = gdf[gdf.geometry.geom_type.isin(['LineString', 'MultiLineString'])]
    if lines.empty:
        print('No line geometries to display.')
        return

    bds    = lines.total_bounds
    center = [(bds[1]+bds[3])/2, (bds[0]+bds[2])/2]
    m      = folium.Map(location=center, zoom_start=zoom, tiles=tiles)

    folium.GeoJson(
        lines.__geo_interface__,
        style_function=lambda feat: {
            'color': CONGESTION_COLORS.get(
                feat['properties'].get(color_col, 'no data'), '#cccccc'),
            'weight': 3 if feat['properties'].get(color_col, 'no data') != 'no data' else 1.5,
        },
        tooltip=None, popup=None,
    ).add_to(m)
    return m


print('Helper functions loaded:  connect(), list_tables(), get_edges(), get_edge_by_id()')
print('                          get_network_stats(), get_congestion_summary()')
print('                          get_congestion_history(), get_runs()')
print('                          get_boundary_cells(), get_edges_in_cell()')
print('                          get_congestion_by_cell(), plot_edges()')

---
## 1. Connect and inspect the database

In [ ]:
%%time
con = connect()
print('All tables in the database:')
display(list_tables(con))

---
## 2. Network statistics

In [ ]:
%%time
print('Network statistics (driving mode):')
display(get_network_stats(con))

---
## 3. Congestion overview

In [ ]:
%%time
print('Current congestion distribution:')
display(get_congestion_summary(con))

In [ ]:
%%time
# Visualise all edges coloured by congestion
edges = get_edges(con)
print(f'{len(edges):,} edges loaded')
plot_edges(edges)

---
## 4. Filter edges

### By congestion level

In [ ]:
%%time
# Get only heavy and severe congestion edges
heavy = get_edges(con, congestion='heavy')
print(f'Heavy congestion: {len(heavy)} edges  ({heavy["length_m"].sum()/1000:.1f} km)')
display(heavy[['edge_id','highway','name','length_m','congestion']].head(10))

### By road type

In [ ]:
%%time
motorways = get_edges(con, highway='motorway')
print(f'Motorway edges: {len(motorways)}')
plot_edges(motorways)

### By road name

In [ ]:
%%time
ring = get_edges(con, name='Ringvägen')
print(f'Ringvägen: {len(ring)} edges  |  congestion: {ring["congestion"].value_counts().to_dict()}')
plot_edges(ring)

---
## 5. Congestion history (time series)

The `edge_congestion_history` table records the congestion value for every edge
every time notebooks 3 + 4 are run. Run the pipeline multiple times (e.g. morning
and evening) to see how congestion changes.

In [ ]:
%%time
print('Pipeline runs recorded:')
display(get_runs(con))

In [ ]:
%%time
# Congestion history for a specific road
history = get_congestion_history(con, road_name='Ringvägen')
if history.empty:
    print('No history yet — run notebooks 3 + 4 at least once.')
else:
    print(f'{len(history)} history rows for Ringvägen:')
    display(history[['run_id','fetched_at','edge_id','name','congestion']])

---
## 6. H3 spatial queries

Each edge has `from_cell` and `to_cell` H3 indices that link it to the boundary hex grid.

In [ ]:
%%time
cells = get_boundary_cells(con)
print(f'{len(cells)} H3 cells at resolution {cells["resolution"].iloc[0]}')

# Congestion per cell
cell_cong = get_congestion_by_cell(con)
print(f'\nCongestion by H3 cell (top 10 by edge count):')
display(cell_cong.head(10))

In [ ]:
%%time
# Query all edges in a specific H3 cell
sample_cell = cell_cong.iloc[0]['h3_id']   # most-connected cell
cell_edges  = get_edges_in_cell(con, sample_cell)
print(f'Cell {sample_cell}: {len(cell_edges)} edges')
plot_edges(cell_edges)

---
## 7. Raw SQL queries

For anything the helper functions don't cover, run SQL directly.
DuckDB supports full SQL including spatial functions.

In [ ]:
%%time
# Example: total network length and cost per mode
con.execute("""
    SELECT 'driving' AS mode,
           count(*)                     AS edges,
           round(sum(length_m)/1000, 1) AS km,
           round(sum(cost_s)/3600, 1)   AS total_travel_hours
    FROM driving.edges
    UNION ALL
    SELECT 'walking', count(*), round(sum(length_m)/1000,1), round(sum(cost_s)/3600,1)
    FROM walking.edges
    UNION ALL
    SELECT 'cycling', count(*), round(sum(length_m)/1000,1), round(sum(cost_s)/3600,1)
    FROM cycling.edges
""").df()

In [ ]:
%%time
# Example: find edges within 500 m of a given point (lon, lat)
LON, LAT = 18.07, 59.315   # somewhere in Södermalm

nearby = con.execute(f"""
    SELECT edge_id, name, highway, congestion,
           round(length_m) AS length_m,
           round(ST_Distance(
               ST_Transform(geometry,    'EPSG:4326', 'EPSG:3857'),
               ST_Transform(ST_Point({LON}, {LAT}), 'EPSG:4326', 'EPSG:3857')
           )) AS dist_m
    FROM driving.edges
    ORDER BY dist_m
    LIMIT 10
""").df()

print(f'10 nearest edges to ({LAT}, {LON}):')
display(nearby)

In [ ]:
con.close()
print('Connection closed.')